# 02 - Pre Processing

After some brief EDA measures, we must then clean our dataset by extracting useful raw features, removing non-negotiable null/empty fields, feature categorization, removing duplicate data, and handling major outliers.

### 02-01 Raw Feature Extraction
__Let's start by only encorperting useful raw features as previously menioned in the EDA notebook.__

The only raw features we wish to keep are as follows:

- **summary**: ticket title or short task description
- **description**: longer issue details
- **priority.name**: priority category
- **labels**: assigned Jira labels
- **subtasks**: linked subtasks
- **votes.votes**: number of votes
- **issuetype.description**: description of the issue type
- **issuetype.name**: issue type label
- **issuetype.subtask**: whether the issue is a subtask
- **resolutiondate**: timestamp used to derive duration
- **created**: timestamp used to derive duration
- **watches.watchCount**: number of watchers

In [10]:
import pandas as pd

ticket_df = pd.read_csv('../jira_ticket_dataset.csv')
pd.set_option('display.max_rows', None)

raw_feature_columns = [
    "summary",
    "description",
    "priority.name",
    "labels",
    "subtasks",
    "votes.votes",
    "issuetype.description",
    "issuetype.name",
    "issuetype.subtask",
    "resolutiondate",
    "created",
    "watches.watchCount"
]

# Reassign dataframe to contain only specified raw features

ticket_df = ticket_df[raw_feature_columns]

ticket_df.head()

C:\Users\Omar\AppData\Local\Temp\ipykernel_14804\4028756587.py:3: DtypeWarning: Columns (0: customfield_12310921, 1: issuetype.subtask) have mixed types. Specify dtype option on import or set low_memory=False.
  ticket_df = pd.read_csv('../jira_ticket_dataset.csv')


,summary,description,priority.name,labels,subtasks,votes.votes,issuetype.description,issuetype.name,issuetype.subtask,resolutiondate,created,watches.watchCount
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,[],[],0.0,An improvement or enhancement to an existing f...,Improvement,False,2005-01-01 07:50:46,2005-01-01 07:47:50,0.0
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors:\n1- runConfigure and conf...,Blocker,[],[],0.0,A problem which impairs or prevents the functi...,Bug,False,2004-12-30 05:30:36,2004-12-25 22:50:30,1.0
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,[],[],0.0,A problem which impairs or prevents the functi...,Bug,False,2005-01-02 15:21:00,2005-01-01 13:52:46,0.0
3,LogHandler can only work in GlobalConfiguratio...,org.apache.axis.handlers.LogHandler in request...,Major,[],[],0.0,A problem which impairs or prevents the functi...,Bug,False,NaN,2005-01-02 19:13:37,0.0
4,Decoding of service is broken in org.apache.ax...,The following code assumes a lot of things:\n\...,Major,[],[],0.0,A problem which impairs or prevents the functi...,Bug,False,NaN,2005-01-03 03:34:52,1.0


#### 02-02 Handling missing / null values

There exists some non-negotiable raw features that **must** be present in order to continue futher analysis such as:

- **summary**: The model cannot predict ticket duration without a summary/description.
- **created**: You cannot calculate task duration without a start date.
- **resolutiondate**: You cannot calculate task duration without an end date.

Therefore we must drop any null rows associated with theses columns.

In [11]:
ticket_df = ticket_df.dropna(subset=["created", "resolutiondate", "summary"])

# check null values of each column to check they are all non-null values

if ticket_df["summary"].isnull().any() or ticket_df["created"].isnull().any() or ticket_df["resolutiondate"].isnull().any():
    print("failed: null value detected for some non-negotioable feature")
else:
    print("success: all non-negotiable features are possess non-null values")

success: all non-negotiable features are possess non-null values


#### 02-03 Feature Categorization

Features are differentiated by multiple factors such as dtype, model learning interpretation, and grouping togther related features.

There will be 3 main feature categories as follows:

- **Text-Based Features**: Descriptive data such as a ticket's description or summary.
- **Numerical-Based Features**: Numerical data such as votes and watch count.
- **summary**: The model cannot predict ticket duration without a summary/description.
